# 05 — Menyiapkan profil 67 kompetensi

Pada tahap ini saya menyiapkan deskripsi kerja bagi setiap kompetensi yang akan digunakan sebagai query pencarian. Sumber awal hanya menyertakan nama dan kata pencarian, sehingga deskripsi tambahan tidak boleh ditulis seolah-olah merupakan definisi resmi BPJS.

Untuk menjaga transparansi, definisi resmi yang tidak tersedia tetap kosong. Query dasar dibuat dari nama kompetensi dan istilah pencarian yang tersedia. Jika dokumen indikator resmi telah diperoleh, kolom profil dapat dilengkapi dan eksperimen dijalankan ulang.

## 1. Membaca sumber data

In [1]:
from pathlib import Path
import re, html, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'data' / 'dataset_buku_2000.csv').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'dataset_buku_2000.csv').exists(), 'Buka folder utama proyek atau notebook dari subfolder notebooks.'
DATA = ROOT / 'data'
OUT = ROOT / 'outputs'
FIG = ROOT / 'figures'
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)
books = pd.read_csv(DATA / 'dataset_buku_2000.csv', dtype=str).fillna('')
comps = pd.read_csv(DATA / 'competencies_green.csv', dtype=str).fillna('')
print(f'Buku: {len(books):,} | Kompetensi: {len(comps)}')

Buku: 973 | Kompetensi: 67


## 2. Periksa seluruh daftar kompetensi

In [2]:
display(comps[['competency_id','excel_column','competency_name','search_query']].head(12))
print('Jumlah:',len(comps),'| Nama unik:',comps.competency_name.nunique())
assert comps.competency_id.is_unique

,competency_id,excel_column,competency_name,search_query
0,C001,AB,Business Development,Business Development
1,C002,AC,Marketing Management,Marketing Management
2,C003,AD,Service and Complaint Management,Service and Complaint Management
3,C004,AE,Membership Administration Management,Membership Administration Management
4,C005,AF,Strategic Purchasing,Strategic Purchasing
5,C006,AG,Provider Management,Provider Management
6,C007,AH,Claim Management,Claim Management
7,C008,AI,Utilization Management,Utilization Management
8,C009,AJ,Fraud Management,Fraud Management
9,C010,AK,Corporate Budgeting Management,Corporate Budgeting Management


Jumlah: 67 | Nama unik: 67


## 3. Gabungkan template profil kerja dan nama referensi

In [3]:
profil_file=DATA/'profil_kompetensi.csv'
if profil_file.exists():
    tambahan=pd.read_csv(profil_file,dtype=str).fillna('')
else:
    tambahan=comps[['competency_id','competency_name']].copy()
for kol in ['definisi_resmi','indikator_perilaku','topik_pembelajaran']:
    if kol not in tambahan.columns: tambahan[kol]=''
profil=comps.merge(tambahan[['competency_id','definisi_resmi','indikator_perilaku','topik_pembelajaran']],on='competency_id',how='left').fillna('')
print('Profil dengan definisi resmi:',int(profil.definisi_resmi.str.strip().ne('').sum()))
display(profil[['competency_name','definisi_resmi','indikator_perilaku']].head(5))

Profil dengan definisi resmi: 0


,competency_name,definisi_resmi,indikator_perilaku
0,Business Development,,
1,Marketing Management,,
2,Service and Complaint Management,,
3,Membership Administration Management,,
4,Strategic Purchasing,,


## 4. Membentuk query yang konsisten
Ini merupakan query kerja untuk eksperimen, bukan definisi kompetensi yang tervalidasi.

In [4]:
usulan = pd.read_csv(DATA/'profil_kerja_usulan_67.csv',dtype=str).fillna('')
profil = profil.merge(usulan,on='competency_id',how='left',validate='one_to_one').fillna('')

def buat_query(row):
    # Prioritas pada profil resmi jika suatu saat tersedia.
    # Topik kerja usulan dipakai untuk memperkaya query bahasa Inggris.
    bagian=[row['competency_name'],row['search_query'],row['topik_pembelajaran_usulan'],
            row['istilah_terkait'],row['definisi_resmi'],row['indikator_perilaku']]
    unik=[]
    for text in bagian:
        text=str(text).strip()
        if text and text not in unik:
            unik.append(text)
    return ' '.join(unik)

profil['teks_query']=profil.apply(buat_query,axis=1)
profil['status_definisi']=np.where(profil.definisi_resmi.str.strip().ne(''),
   'Definisi sumber tersedia', 'USULAN EKSPERIMEN — definisi resmi belum tersedia')
display(profil[['competency_id','competency_name','definisi_kerja_usulan','topik_pembelajaran_usulan','teks_query']].head(6))

,competency_id,competency_name,definisi_kerja_usulan,topik_pembelajaran_usulan,teks_query
0,C001,Business Development,Kemampuan mengelola dan menerapkan praktik Bus...,business growth; business strategy; market opp...,Business Development business growth; business...
1,C002,Marketing Management,Kemampuan mengelola dan menerapkan praktik Mar...,marketing strategy; marketing analytics; custo...,Marketing Management marketing strategy; marke...
2,C003,Service and Complaint Management,Kemampuan mengelola dan menerapkan praktik Ser...,customer service; complaint handling; service ...,Service and Complaint Management customer serv...
3,C004,Membership Administration Management,Kemampuan mengelola dan menerapkan praktik Mem...,membership administration; eligibility; enroll...,Membership Administration Management membershi...
4,C005,Strategic Purchasing,Kemampuan mengelola dan menerapkan praktik Str...,strategic purchasing; value based healthcare; ...,Strategic Purchasing strategic purchasing; val...
5,C006,Provider Management,Kemampuan mengelola dan menerapkan praktik Pro...,healthcare provider networks; provider contrac...,Provider Management healthcare provider networ...


## 5. Periksa kelengkapan dan simpan

In [5]:
assert len(profil)==len(comps)
assert profil.competency_id.is_unique
assert profil.teks_query.str.strip().ne('').all()
profil.to_csv(OUT/'profil_kompetensi_query.csv',index=False,encoding='utf-8-sig')
print('Query tersimpan:',OUT/'profil_kompetensi_query.csv')
print('Deskripsi resmi belum tersedia:',int(profil.definisi_resmi.eq('').sum()))

Query tersimpan: c:\Users\AUREL ELVIOLITA\Downloads\BPJS_Book_Recommendation\Book_Recommendation\outputs\profil_kompetensi_query.csv
Deskripsi resmi belum tersedia: 67


## Kesimpulan tahap
Seluruh 67 kompetensi mempunyai **deskripsi kerja usulan, kemampuan, topik pembelajaran, dan istilah terkait**. File `data/profil_kerja_usulan_67.csv` memperlihatkan asal istilah tambahan. Ini **bukan definisi resmi BPJS**; istilah perlu ditinjau mentor dan dikoreksi sesuai dokumen kompetensi resmi jika tersedia. Semua model nantinya menggunakan query yang sama.